# Capítulo 1 — Datos (ETL)

**Objetivo:** precios ajustados → series mensuales → retornos.

- Precio ajustado ≈ total return.
- $R_t = P_t/P_{t-1}-1$
- Ventana: ene-2016 → sep-2026 (necesitamos 2015-12).


In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT/'src'))
import portfolio_utils as pu
tickers = sorted(set(list(pu.ORIGINAL_WEIGHTS)+list(pu.CURRENT_WEIGHTS)+[pu.BENCHMARK]))
print('Tickers:', tickers)


Tickers: ['BRK-B', 'IEMG', 'SMH', 'SPY', 'SPYG', 'VTI']


In [2]:
csv = ROOT/'data'/'precios_ajustados_diarios.csv'
daily = pd.read_csv(csv, index_col=0, parse_dates=True)
print('Leído desde cache:', csv.name)
monthly = pu.to_month_end(daily).loc[:'2026-09-30']
print(monthly.index.min().date(), '→', monthly.index.max().date(), '| meses:', len(monthly))
rets = monthly.pct_change().loc['2016-01-31':'2026-09-30']
print('Retornos shape:', rets.shape)
rf = pu.load_risk_free_monthly()
print('RF media mensual (~T-Bill):', round(rf.mean()*100, 4), '%')


Leído desde cache: data/precios_ajustados_diarios.csv
2015-11-30 → 2026-09-30 | meses: 131
Retornos shape: (129, 6)
RF media mensual (~T-Bill): 0.1889 %


In [3]:
norm = monthly.loc['2016-01-31':, list(pu.CURRENT_WEIGHTS)].dropna()
norm = norm / norm.iloc[0] * 100
ax = norm.plot(figsize=(11,5), title='Precios normalizados (base 100)')
ax.set_ylabel('Índice'); plt.tight_layout(); plt.show()


![figura](../figures/nb01_precios_norm.svg)


**Interpretación:** `SMH` suele ser el más explosivo; `IEMG` quedó atrás 2016–2024 vs EE.UU. growth.

## Conclusiones
- ETL reproducible vía `portfolio_utils.download_prices`.
- Frecuencia mensual alineada con Portfolio Visualizer.

## Ejercicios
1. Compara IEMG vs SPYG en 2022–2023.
2. Correlación VFINX vs SPY.
3. `resample('BM')` vs `'ME'`.
